## Transform Constructors Data
1. Read bronze constructors table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId → constructor_id)
4. Rename columns to make them more meaningful (name → constructor_name)
5. Remove duplicate records
6. Transform values of column nationality to Title Case
7. Write the transformed data to silver constructors table

In [0]:
# 1. Get batch ID
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/03_silver_helpers"

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_path = "s3://bizf1project123/Bronze/Kalashri_increment/constructors"

silver_path = "s3://bizf1project123/Silver/Kalashri_increment/constructors"

## Step 1 - Read bronze constructors table

In [0]:
# 5. Read Bronze Delta data for the selected batch
constructors_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
    .filter(F.col("batch_id") == v_batch_id)
)

display(constructors_df)

## Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
constructors_dropped_df = constructors_df.drop("url")

## Step 3 & 4 - Standardise Column Names
-    Standardise column names using snake_case (constructorId → constructor_id)
-    Rename columns to make them more meaningful (name → constructor_name)

In [0]:
constructors_renamed_df = (
    constructors_dropped_df
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "name": "constructor_name"
        })
)

display(constructors_renamed_df)

## Step 5 - Remove duplicate records


In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

display(constructors_distinct_df)

## Step 6 - Transform values of column nationality to Title Case

In [0]:
constructors_final_df = (
    constructors_distinct_df
        .withColumn('nationality', F.initcap(F.col("nationality")))
)

display(constructors_final_df)


## Step 7 - Write the transformed data to silver constructors table

In [0]:
write_to_silver(
    input_df=constructors_final_df,
    target_path=silver_path,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
%sql
SELECT *
FROM delta.`s3://bizf1project123/Silver/Kalashri_increment/constructors`;

In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Silver/Kalashri_increment/constructors",
#     True
# )